# Model Comparison: Predicting Midfielder Post-Transfer Qualities

Five OLS regression models of increasing complexity. All evaluated on the **same target**: `to_Q_i` (post-transfer quality level).
Models 4-5 internally predict the delta, then reconstruct `to_Q_i = from_Q_i + predicted_delta` for a fair comparison.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error

# ── Load & prepare ──────────────────────────────────────────────────────

DATA_DIR = "../../../../thesis_data/processed_data/thesis_model_dataset/active/"
df = pd.read_parquet(DATA_DIR + "within_league_transfers_v5.parquet")
mf = df[df["from_position"] == "Midfielder"].copy()

QUALITIES = [
    "Active defence", "Aerial threat", "Box threat", "Composure",
    "Defensive heading", "Dribbling", "Effectiveness", "Finishing",
    "Hold-up play", "Intelligent defence", "Involvement",
    "Passing quality", "Pressing", "Progression",
    "Providing teammates", "Run quality", "Winning duels",
]

TEAM_Q = ["DEFENCE", "DEFENSIVE_TRANSITION", "ATTACKING_TRANSITION",
          "ATTACK", "PENETRATION", "CHANCE_CREATION", "OUTCOME"]

from_pq = [f"from_{q}" for q in QUALITIES]
to_pq   = [f"to_{q}" for q in QUALITIES]
from_tq = [f"from_q_proj_{q}" for q in TEAM_Q]
to_tq   = [f"to_q_{q}" for q in TEAM_Q]

for q in TEAM_Q:
    mf[f"delta_team_{q}"] = mf[f"to_q_{q}"] - mf[f"from_q_proj_{q}"]
delta_tq = [f"delta_team_{q}" for q in TEAM_Q]

all_cols = from_pq + to_pq + from_tq + to_tq + delta_tq + ["from_season"]
mf_clean = mf[all_cols].dropna()

train = mf_clean[mf_clean["from_season"] <= 2023]
test  = mf_clean[mf_clean["from_season"] == 2024]

print(f"Train: {len(train):,} | Test: {len(test):,}")

Train: 4,383 | Test: 465


In [2]:
def run_model(train, test, features_fn, target_fn, model_name, delta_mode=False):
    """
    Run 17 OLS regressions (one per quality).
    
    If delta_mode=True, the model predicts delta_Q internally,
    but R2 and MAE are measured on to_Q (= from_Q + predicted_delta).
    """
    rows = []
    for q in QUALITIES:
        feat_cols = features_fn(q)
        target_col = target_fn(q)

        X_train = sm.add_constant(train[feat_cols])
        X_test  = sm.add_constant(test[feat_cols])
        y_train = train[target_col]
        y_test_internal = test[target_col]

        model = sm.OLS(y_train, X_train).fit()
        y_pred_internal = model.predict(X_test)

        # Always evaluate on to_Q
        y_test_actual = test[f"to_{q}"]
        if delta_mode:
            y_pred_to = test[f"from_{q}"] + y_pred_internal
        else:
            y_pred_to = y_pred_internal

        ss_res = ((y_test_actual - y_pred_to) ** 2).sum()
        ss_tot = ((y_test_actual - y_test_actual.mean()) ** 2).sum()
        r2_test = 1 - ss_res / ss_tot

        rows.append({
            "Quality": q,
            "R2_train": model.rsquared,
            "R2_adj_train": model.rsquared_adj,
            "R2_test": r2_test,
            "MAE_test": mean_absolute_error(y_test_actual, y_pred_to),
            "F_pvalue": model.f_pvalue,
            "N_features": len(feat_cols),
        })

    results = pd.DataFrame(rows)
    results["Model"] = model_name
    return results

---
## Model 1: Naive Baseline

$$\text{to\_Q}_i = \alpha + \beta \cdot \text{from\_Q}_i$$

Simplest hypothesis: a player's post-transfer quality is a linear function of that same quality before the transfer.

In [3]:
m1 = run_model(
    train, test,
    features_fn=lambda q: [f"from_{q}"],
    target_fn=lambda q: f"to_{q}",
    model_name="1. Naive Baseline",
)
m1.sort_values("R2_test", ascending=False).head(5).round(4)

,Quality,R2_train,R2_adj_train,R2_test,MAE_test,F_pvalue,N_features,Model
11,Passing quality,0.4703,0.4701,0.4472,0.4087,0.0,1,1. Naive Baseline
13,Progression,0.4311,0.4310,0.3796,0.4920,0.0,1,1. Naive Baseline
15,Run quality,0.4699,0.4698,0.3682,0.3157,0.0,1,1. Naive Baseline
14,Providing teammates,0.3860,0.3858,0.3596,0.4201,0.0,1,1. Naive Baseline
4,Defensive heading,0.3742,0.3741,0.3587,0.4913,0.0,1,1. Naive Baseline


---
## Model 2: Player Profile

$$\text{to\_Q}_i = \alpha + \sum_{j=1}^{17} \beta_j \cdot \text{from\_Q}_j$$

All 17 pre-transfer qualities predict each post-transfer quality. Captures cross-quality effects.

In [4]:
m2 = run_model(
    train, test,
    features_fn=lambda q: from_pq,
    target_fn=lambda q: f"to_{q}",
    model_name="2. Player Profile",
)
m2.sort_values("R2_test", ascending=False).head(5).round(4)

,Quality,R2_train,R2_adj_train,R2_test,MAE_test,F_pvalue,N_features,Model
11,Passing quality,0.4916,0.4896,0.4641,0.4024,0.0,17,2. Player Profile
13,Progression,0.4520,0.4499,0.4119,0.4745,0.0,17,2. Player Profile
4,Defensive heading,0.4231,0.4208,0.4036,0.4753,0.0,17,2. Player Profile
15,Run quality,0.4955,0.4935,0.4004,0.3103,0.0,17,2. Player Profile
14,Providing teammates,0.4380,0.4358,0.4001,0.4045,0.0,17,2. Player Profile


---
## Model 3: Player + Team Context

$$\text{to\_Q}_i = \alpha + \sum_{j=1}^{17} \beta_j \cdot \text{from\_Q}_j + \sum_{k=1}^{7} \gamma_k \cdot \text{from\_team}_k + \sum_{k=1}^{7} \delta_k \cdot \text{to\_team}_k$$

Adds origin and destination team tactical styles. Tests whether team context explains variance beyond the player's own qualities.

In [5]:
m3 = run_model(
    train, test,
    features_fn=lambda q: from_pq + from_tq + to_tq,
    target_fn=lambda q: f"to_{q}",
    model_name="3. Player + Team Context",
)
m3.sort_values("R2_test", ascending=False).head(5).round(4)

,Quality,R2_train,R2_adj_train,R2_test,MAE_test,F_pvalue,N_features,Model
11,Passing quality,0.5454,0.5422,0.5210,0.3876,0.0,31,3. Player + Team Context
10,Involvement,0.4658,0.4620,0.4935,0.3256,0.0,31,3. Player + Team Context
14,Providing teammates,0.4654,0.4616,0.4384,0.3908,0.0,31,3. Player + Team Context
13,Progression,0.4777,0.4740,0.4360,0.4709,0.0,31,3. Player + Team Context
15,Run quality,0.5162,0.5128,0.4210,0.3061,0.0,31,3. Player + Team Context


---
## Model 4: Tactical Shift

$$\Delta PQ_i = \alpha + \sum_{k=1}^{7} \gamma_k \cdot \Delta TQ_k$$

Can the change in tactical environment alone explain post-transfer quality? No player baseline — purely team-driven.

Internally predicts `delta_Q`, evaluated as `to_Q = from_Q + predicted_delta`.

In [6]:
m4 = run_model(
    train, test,
    features_fn=lambda q: delta_tq,
    target_fn=lambda q: f"to_{q}",
    model_name="4. Tactical Shift",
    delta_mode=False,
)

# Special case: this model predicts to_Q directly using only delta_tq
# (predicting delta and adding from_Q would be equivalent to including from_Q as feature with coef=1)
# So we keep it as direct prediction for honest evaluation
m4.sort_values("R2_test", ascending=False).head(5).round(4)

,Quality,R2_train,R2_adj_train,R2_test,MAE_test,F_pvalue,N_features,Model
10,Involvement,0.0477,0.0462,0.0375,0.4609,0.0,7,4. Tactical Shift
6,Effectiveness,0.0303,0.0287,0.0355,0.2747,0.0,7,4. Tactical Shift
12,Pressing,0.0146,0.0130,0.0206,0.5374,0.0,7,4. Tactical Shift
3,Composure,0.0273,0.0257,0.0200,0.3896,0.0,7,4. Tactical Shift
14,Providing teammates,0.0144,0.0129,0.0136,0.5255,0.0,7,4. Tactical Shift


---
## Model 5: Player + Tactical Shift

$$\Delta PQ_i = \alpha + \sum_{j=1}^{17} \beta_j \cdot \text{from\_Q}_j + \sum_{k=1}^{7} \gamma_k \cdot \Delta TQ_k$$

Combines the player's baseline with the change in tactical environment.

Internally predicts `delta_Q`, evaluated as `to_Q = from_Q + predicted_delta`.

In [7]:
# Build delta targets for training
train_delta = train.copy()
test_delta = test.copy()
for q in QUALITIES:
    train_delta[f"delta_{q}"] = train_delta[f"to_{q}"] - train_delta[f"from_{q}"]
    test_delta[f"delta_{q}"] = test_delta[f"to_{q}"] - test_delta[f"from_{q}"]

m5 = run_model(
    train_delta, test_delta,
    features_fn=lambda q: from_pq + delta_tq,
    target_fn=lambda q: f"delta_{q}",
    model_name="5. Player + Tactical Shift",
    delta_mode=True,
)
m5.sort_values("R2_test", ascending=False).head(5).round(4)

,Quality,R2_train,R2_adj_train,R2_test,MAE_test,F_pvalue,N_features,Model
11,Passing quality,0.2576,0.2535,0.4983,0.3945,0.0,24,5. Player + Tactical Shift
10,Involvement,0.3413,0.3377,0.4469,0.3420,0.0,24,5. Player + Tactical Shift
14,Providing teammates,0.2727,0.2687,0.4264,0.3938,0.0,24,5. Player + Tactical Shift
13,Progression,0.2391,0.2349,0.4240,0.4722,0.0,24,5. Player + Tactical Shift
15,Run quality,0.2119,0.2076,0.4103,0.3093,0.0,24,5. Player + Tactical Shift


---
## Results

### Top 5 best-predicted qualities per model

In [8]:
all_results = pd.concat([m1, m2, m3, m4, m5], ignore_index=True)

top5 = (all_results
        .sort_values(["Model", "R2_test"], ascending=[True, False])
        .groupby("Model")
        .head(5))

display_cols = ["Model", "Quality", "R2_test", "R2_adj_train", "MAE_test", "F_pvalue"]

for model_name, group in top5.groupby("Model", sort=False):
    print(f"\n{'='*70}")
    print(f"  {model_name}")
    print(f"{'='*70}")
    print(group[display_cols].to_string(index=False, float_format="{:.4f}".format))


  1. Naive Baseline
            Model             Quality  R2_test  R2_adj_train  MAE_test  F_pvalue
1. Naive Baseline     Passing quality   0.4472        0.4701    0.4087    0.0000
1. Naive Baseline         Progression   0.3796        0.4310    0.4920    0.0000
1. Naive Baseline         Run quality   0.3682        0.4698    0.3157    0.0000
1. Naive Baseline Providing teammates   0.3596        0.3858    0.4201    0.0000
1. Naive Baseline   Defensive heading   0.3587        0.3741    0.4913    0.0000

  2. Player Profile
            Model             Quality  R2_test  R2_adj_train  MAE_test  F_pvalue
2. Player Profile     Passing quality   0.4641        0.4896    0.4024    0.0000
2. Player Profile         Progression   0.4119        0.4499    0.4745    0.0000
2. Player Profile   Defensive heading   0.4036        0.4208    0.4753    0.0000
2. Player Profile         Run quality   0.4004        0.4935    0.3103    0.0000
2. Player Profile Providing teammates   0.4001        0.4358    0.4

### R² test + significance (all qualities x all models)

In [9]:
r2_pivot = all_results.pivot(index="Quality", columns="Model", values="R2_test").round(4)
pval_pivot = all_results.pivot(index="Quality", columns="Model", values="F_pvalue")

def fmt_cell(r2, pval):
    stars = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    return f"{r2:.4f}{stars}"

combined = pd.DataFrame(index=r2_pivot.index, columns=r2_pivot.columns)
for col in r2_pivot.columns:
    for idx in r2_pivot.index:
        combined.loc[idx, col] = fmt_cell(r2_pivot.loc[idx, col], pval_pivot.loc[idx, col])

print("R2 test (*** p<0.001, ** p<0.01, * p<0.05)")
print("=" * 110)
print(combined.to_string())

R2 test (*** p<0.001, ** p<0.01, * p<0.05)
Model               1. Naive Baseline 2. Player Profile 3. Player + Team Context 4. Tactical Shift 5. Player + Tactical Shift
Quality                                                                                                                      
Active defence              0.3013***         0.3480***                0.3514***         -0.0003**                  0.3525***
Aerial threat               0.1852***         0.2499***                0.2688***          0.0019**                  0.2643***
Box threat                  0.3435***         0.3622***                0.3853***         0.0085***                  0.3826***
Composure                   0.0290***         0.0759***                0.1556***         0.0200***                  0.1305***
Defensive heading           0.3587***         0.4036***                0.4030***           -0.0041                  0.4063***
Dribbling                   0.1896***         0.2053***                0.21

### Best model per quality (p < 0.05)

In [10]:
valid = all_results[all_results["F_pvalue"] < 0.05]
best = valid.loc[valid.groupby("Quality")["R2_test"].idxmax()][["Quality", "Model", "R2_test", "MAE_test", "F_pvalue"]]
best = best.sort_values("R2_test", ascending=False).reset_index(drop=True)

print("Best model per quality")
print("=" * 80)
print(best.to_string(index=False, float_format="{:.4f}".format))

Best model per quality
            Quality                      Model  R2_test  MAE_test  F_pvalue
    Passing quality   3. Player + Team Context   0.5210    0.3876    0.0000
        Involvement   3. Player + Team Context   0.4935    0.3256    0.0000
Providing teammates   3. Player + Team Context   0.4384    0.3908    0.0000
        Progression   3. Player + Team Context   0.4360    0.4709    0.0000
        Run quality   3. Player + Team Context   0.4210    0.3061    0.0000
  Defensive heading 5. Player + Tactical Shift   0.4063    0.4757    0.0000
         Box threat   3. Player + Team Context   0.3853    0.3582    0.0000
           Pressing   3. Player + Team Context   0.3728    0.4340    0.0000
Intelligent defence   3. Player + Team Context   0.3563    0.4426    0.0000
     Active defence 5. Player + Tactical Shift   0.3525    0.4290    0.0000
      Effectiveness   3. Player + Team Context   0.3051    0.2328    0.0000
      Aerial threat   3. Player + Team Context   0.2688    0.4289